# 01 — MLP Baseline

Train a Multi-Layer Perceptron as a weak baseline.
- Flatten 224×224×3 image → dense layers → classifier
- **Not expected to win** — provides a lower bound for comparison

### Expected Output
- Best checkpoint → `ml/artifacts/checkpoints/mlp_best.pth`
- Training curves, confusion matrix, per-class F1
- Classification report and training summary JSON

In [2]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
if 'notebooks' in str(PROJECT_ROOT):
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/ajitkumarsingh/Desktop/desktop/cattle-breed-classifier-webapp


In [3]:
from ml.src.utils.seed import set_seed
from ml.src.utils.device import get_device
from ml.src.utils.io import load_config
from ml.src.utils.manifests import create_dataloaders
from ml.src.data.transforms import get_train_transforms, get_eval_transforms
from ml.src.models.mlp import CattleMLP
from ml.src.training.trainer import Trainer

set_seed(42)
device = get_device()

Using Apple MPS (Metal Performance Shaders)


## 1. Load Config & Data

In [4]:
config = load_config('mlp', PROJECT_ROOT / 'ml' / 'configs')
config['num_classes'] = 26

img_size = config['image']['size']
batch_size = config['training']['batch_size']

print(f'Image size: {img_size}')
print(f'Batch size: {batch_size}')
print(f'Epochs: {config["training"]["num_epochs"]}')
print(f'LR: {config["training"]["learning_rate"]}')

Image size: 224
Batch size: 64
Epochs: 50
LR: 0.001


In [5]:
train_transforms = get_train_transforms(img_size=img_size)
eval_transforms = get_eval_transforms(img_size=img_size)

dataloaders = create_dataloaders(
    manifests_dir=PROJECT_ROOT / 'ml' / 'artifacts' / 'manifests',
    data_root=PROJECT_ROOT / 'Cattle_Resized',
    train_transform=train_transforms,
    eval_transform=eval_transforms,
    batch_size=batch_size,
    num_workers=config['data'].get('num_workers', 4),
)

class_names = dataloaders['train'].dataset.classes
print(f'Classes: {len(class_names)}')
print(f'Train: {len(dataloaders["train"].dataset)}')
print(f'Val: {len(dataloaders["val"].dataset)}')
print(f'Test: {len(dataloaders["test"].dataset)}')

Classes: 26
Train: 2139
Val: 458
Test: 459


## 2. Create Model

In [7]:
model = CattleMLP(hidden_layers=[5]).from_config(config)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')
print(f'Model size: {total_params * 4 / 1024 / 1024:.1f} MB (float32)')

CattleMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (classifier): Sequential(
    (0): Linear(in_features=150528, out_features=1024, bias=True)
    (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=512, out_features=256, bias=True)
    (9): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=256, out_features=26, bias=True)
  )
)

Total parameters: 154,808,090
Model size: 590.5 MB (float32)


## 3. Train

In [ ]:
trainer = Trainer(
    model=model,
    config=config,
    dataloaders=dataloaders,
    class_names=class_names,
    device=device,
    model_name='mlp',
)



Training: mlp
Device: mps
Epochs: 50



/Users/ajitkumarsingh/miniforge3/envs/cattle-classifier/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
python(53489) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(53490) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(53491) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(53492) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  Epoch   1 | Train Loss: 6.1549 | Train Acc: 0.0800 | Val Loss: 3.9679 | Val Acc: 0.1245 | LR: 0.001000
  Checkpoint saved: mlp_best.pth (value: 3.9679)


python(53517) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(53518) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(53519) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(53520) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(53979) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(53980) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(53981) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(53982) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  Epoch   2 | Train Loss: 4.9052 | Train Acc: 0.0890 | Val Loss: 3.8072 | Val Acc: 0.1376 | LR: 0.001000
  Checkpoint saved: mlp_best.pth (value: 3.8072)


...

  Epoch  42 | Train Loss: 2.8213 | Train Acc: 0.2107 | Val Loss: 2.6606 | Val Acc: 0.2183 | LR: 0.000010
  Checkpoint saved: mlp_best.pth (value: 2.6606)


python(59053) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(59054) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(59055) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(59056) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(59184) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(59185) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(59186) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(59187) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  Epoch  43 | Train Loss: 2.8989 | Train Acc: 0.2188 | Val Loss: 2.6590 | Val Acc: 0.2118 | LR: 0.000010
  Checkpoint saved: mlp_best.pth (value: 2.6590)


python(59238) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(59240) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(59241) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(59242) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


KeyboardInterrupt: 

## 4. Evaluate on Test Set

In [9]:
test_results = trainer.evaluate(split='test')


Evaluating: mlp on test

Loaded best checkpoint from epoch 43


python(72795) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(72796) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(72797) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(72798) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



TEST Results:
  Loss:          2.6458
  Accuracy:      0.2375
  Macro F1:      0.1197
  Macro Prec:    0.1291
  Macro Recall:  0.1481
  Avg Latency:   0.53 ms
  Model Size:    590.55 MB


## 5. Save Artifacts

In [ ]:
trainer.save_artifacts()

print('\n=== MLP Baseline Summary ===')
print(f'Best Val Loss:  {training_summary["best_val_loss"]:.4f}')
print(f'Best Val Acc:   {training_summary["best_val_accuracy"]:.4f}')
print(f'Test Accuracy:  {test_results["metrics"]["accuracy"]:.4f}')
print(f'Test Macro F1:  {test_results["metrics"]["macro_f1"]:.4f}')
print(f'Latency:        {test_results["latency"]["avg_ms"]:.2f} ms')
print(f'Model Size:     {test_results["model_size_mb"]:.2f} MB')


Saving artifacts for mlp...
  Saved training curves to /Users/ajitkumarsingh/Desktop/desktop/cattle-breed-classifier-webapp/ml/artifacts/figures/mlp/training_curves.png


/Users/ajitkumarsingh/miniforge3/envs/cattle-classifier/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
python(72814) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(72815) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(72816) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(72817) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  Saved confusion matrix to /Users/ajitkumarsingh/Desktop/desktop/cattle-breed-classifier-webapp/ml/artifacts/figures/mlp/confusion_matrix.png
  Saved per-class F1 chart to /Users/ajitkumarsingh/Desktop/desktop/cattle-breed-classifier-webapp/ml/artifacts/figures/mlp/per_class_f1.png
  Artifacts saved to /Users/ajitkumarsingh/Desktop/desktop/cattle-breed-classifier-webapp/ml/artifacts/figures/mlp

=== MLP Baseline Summary ===


NameError: name 'training_summary' is not defined

## 6. Classification Report

In [12]:
print(test_results['metrics']['classification_report'])

                    precision    recall  f1-score   support

      Alambadi Cow       0.00      0.00      0.00        14
    Amritmahal Cow       0.00      0.00      0.00        14
     Banni Buffalo       0.00      0.00      0.00         5
        Bargur Cow       0.00      0.00      0.00         9
         Dangi Cow       0.00      0.00      0.00        12
         Deoni Cow       0.00      0.00      0.00        16
           Gir Cow       0.32      0.53      0.40        38
      Hallikar Cow       0.27      0.43      0.33        28
Jaffrabadi Buffalo       0.28      0.31      0.29        16
      Kangayam Cow       0.50      0.06      0.10        18
       Kankrej Cow       0.19      0.19      0.19        27
     Kasaragod Cow       0.00      0.00      0.00        14
      Kenkatha Cow       0.00      0.00      0.00         8
     Kherigarh Cow       0.00      0.00      0.00         5
  Malnad gidda Cow       0.17      0.25      0.20        16
   Mehsana Buffalo       0.22      0.27

---
**✅ MLP Baseline complete.** Proceed to `02_cnn_from_scratch.ipynb`.